#### Using python [sqlframe](https://github.com/eakmanrq/sqlframe) library to use PySpark dataframe API against a PostgreSQL database

In [1]:
from dotenv import load_dotenv
import os
from psycopg2 import connect
from sqlframe.postgres import functions as F
from sqlframe.postgres import PostgresSession
from sqlframe.postgres import Window

In [2]:
load_dotenv()

conn = connect(
    dbname=os.environ["DB_NAME"],
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    host=os.environ["DB_HOST"],
    port=os.environ["DB_PORT"],
)

"""One caveat: autocommit mode means each statement is its own transaction, so if a multi-statement write ever needs to be atomic
(e.g., you want the drop and create to succeed or fail together), you'd lose that guarantee. For a single saveAsTable() call this isn't a concern,
but keep it in mind if your script grows."""
conn.autocommit = True  # ensures DDL/writes are visible to other connections immediately

session = PostgresSession(conn=conn)

#### Reading a PostgreSQL table as a PySpark dataframe

In [3]:
df = (
    session.table('public.vehicles')
    .where(F.col("year") == "2027")
    .select("year","make","model","fueltype","fueltype1","fueltype2")
)

In [4]:
df.printSchema()

root
 |-- year: int (nullable = true)
 |-- make: string (nullable = true)
 |-- model: string (nullable = true)
 |-- fueltype: string (nullable = true)
 |-- fueltype1: string (nullable = true)
 |-- fueltype2: string (nullable = true)


sqlframe has features that PySpark does not have like `saveAsTable()` method, which allows you to save a dataframe as a table with just one line of code.  Unfortunately, there is currently a bug where if using PostgreSQL as the backend, the `saveAsTable()` method will fail because the underlying SQL that sqlframe generates is using a dialect using `CREATE OR REPLACE TABLE` syntax which PostgreSQL does not support.

In [ ]:
df.write.mode("overwrite").saveAsTable("my_new_table")
conn.commit()

**Workaround:** Execute with correct SQL statements using connection cursor

In [5]:
# Drop the target table first (if it exists), since sqlframe's
# mode("overwrite") emits "CREATE OR REPLACE TABLE", which Postgres
# doesn't support.
with conn.cursor() as cur:
    cur.execute('DROP TABLE IF EXISTS public.models_2027')

# No mode() needed now — the table doesn't exist, so this generates
# a plain CREATE TABLE ... AS SELECT
df.write.saveAsTable('public.models_2027')

#### Let's check that our new table was actually made and has data in it

In [6]:
table = session.table("public.models_2027")
table.limit(5).show()

+------+----------+-------------------+----------+------------------+-----------+
| year |   make   |       model       | fueltype |    fueltype1     | fueltype2 |
+------+----------+-------------------+----------+------------------+-----------+
| 2027 | Chrysler |    Pacifica AWD   | Regular  | Regular Gasoline |           |
| 2027 |  Lotus   |       Emira       | Regular  | Regular Gasoline |           |
| 2027 |   BMW    |     430i Coupe    | Premium  | Premium Gasoline |           |
| 2027 |   BMW    | 430i xDrive Coupe | Premium  | Premium Gasoline |           |
| 2027 |   BMW    |  430i Convertible | Premium  | Premium Gasoline |           |
+------+----------+-------------------+----------+------------------+-----------+


#### We can also use `listTables()` to obtain a list of tables

In [7]:
tables = session.catalog.listTables()

In [8]:
type(tables)

list

In [9]:
for table in tables:
    print(table)

Table(name='vehicles', catalog='postgres', namespace=['public'], description=None, tableType='MANAGED', isTemporary=False)
Table(name='models_2027', catalog='postgres', namespace=['public'], description=None, tableType='MANAGED', isTemporary=False)


#### Some filtering examples

In [10]:
df_elec = (
    session.table('public.vehicles')
    .filter(
        (F.col("fueltype").like("%Elec%")) &
        (F.col("year") == 2027)
    )
    .select("year", "make", "model", "fueltype", "fueltype1", "fueltype2", "startstop", "atvtype")
)

df_elec.limit(5).show()

+------+------+------------------------------------+-------------+-------------+-----------+-----------+---------+
| year | make |               model                |   fueltype  |  fueltype1  | fueltype2 | startstop | atvtype |
+------+------+------------------------------------+-------------+-------------+-----------+-----------+---------+
| 2027 | BMW  | i5 eDrive40 Sedan (19 inch Wheels) | Electricity | Electricity |           |     N     |    EV   |
| 2027 | BMW  | i5 eDrive40 Sedan (20 inch Wheels) | Electricity | Electricity |           |     N     |    EV   |
| 2027 | BMW  | i5 eDrive40 Sedan (21 inch Wheels) | Electricity | Electricity |           |     N     |    EV   |
| 2027 | BMW  | i5 xDrive40 Sedan (20 inch Wheels) | Electricity | Electricity |           |     N     |    EV   |
| 2027 | BMW  | i5 xDrive40 Sedan (21 inch Wheels) | Electricity | Electricity |           |     N     |    EV   |
+------+------+------------------------------------+-------------+-------------+

In [11]:
df_gas = (
    session.table('public.vehicles')
    .filter(
        (F.col("fueltype1").like("%Gas%"))
    )
    .select("year", "make", "model", "fueltype", "fueltype1", "fueltype2", "startstop", "atvtype", "comb08", "highway08")
)

df_gas.limit(5).show()

+------+------------+---------------------+----------+------------------+-----------+-----------+---------+--------+-----------+
| year |    make    |        model        | fueltype |    fueltype1     | fueltype2 | startstop | atvtype | comb08 | highway08 |
+------+------------+---------------------+----------+------------------+-----------+-----------+---------+--------+-----------+
| 1985 | Alfa Romeo |  Spider Veloce 2000 | Regular  | Regular Gasoline |           |           |         |   21   |     25    |
| 1985 |  Ferrari   |      Testarossa     | Regular  | Regular Gasoline |           |           |         |   11   |     14    |
| 1985 |   Dodge    |       Charger       | Regular  | Regular Gasoline |           |           |         |   27   |     33    |
| 1985 |   Dodge    | B150/B250 Wagon 2WD | Regular  | Regular Gasoline |           |           |         |   11   |     12    |
| 1993 |   Subaru   |   Legacy AWD Turbo  | Premium  | Premium Gasoline |           |           |

In [12]:
df_gas.count()

47316

#### ICE, non-hybrid vehicles with highest highway08 (highway MPG) for each model year

In [13]:
window_spec = Window.partitionBy("year").orderBy(F.col("highway08").desc())

df_gas.filter(
    (F.col("year") >= 2010) &
    (F.col("year") != 2027) &
    (~F.col("atvtype").like("%Hybrid%")) &
    (~F.col("model").like("%Hybrid%")) &
    (~F.col("atvtype").like("%FFV%")) &
    (~F.col("model").like("%FFV%"))
).withColumn(
    "yr_rank", F.dense_rank().over(window_spec)
).filter(
    F.col("yr_rank") <= 1
).select(
    "year",
    "make",
    "model",
    "yr_rank",
    "highway08",
    "fueltype",
    "fueltype1",
    "fueltype2",
    "atvtype"
).orderBy(
    F.col("year").desc(),
    F.col("yr_rank").asc()
).show()

+------+------------+---------------------+---------+-----------+----------+------------------+-----------+---------+
| year |    make    |        model        | yr_rank | highway08 | fueltype |    fueltype1     | fueltype2 | atvtype |
+------+------------+---------------------+---------+-----------+----------+------------------+-----------+---------+
| 2026 |   Toyota   | Corolla (1-mode TM) |    1    |     41    | Regular  | Regular Gasoline |           |         |
| 2026 |   Honda    |      Civic 4Dr      |    1    |     41    | Regular  | Regular Gasoline |           |         |
| 2026 |   Toyota   |  Corolla Hatchback  |    1    |     41    | Regular  | Regular Gasoline |           |         |
| 2025 |   Toyota   |  Corolla Hatchback  |    1    |     41    | Regular  | Regular Gasoline |           |         |
| 2025 |   Honda    |      Civic 4Dr      |    1    |     41    | Regular  | Regular Gasoline |           |         |
| 2025 |  Hyundai   |       Elantra       |    1    |   